# Data profile

Profiles the raw IEEE-CIS CSVs. Uses `data/raw` when the Kaggle download is present, otherwise the synthetic sample in `data/raw-sample` (sample numbers are not results).

The same logic is in `prep/profile_data.py`, so the numbers here match `docs/data_profile.md`.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "prep" else Path.cwd()
sys.path.insert(0, str(ROOT / "prep"))
import pandas as pd
import matplotlib.pyplot as plt
from profile_data import load_raw, profile, to_markdown, DAY

REAL = ROOT / "data/raw"
RAW = REAL if (REAL / "train_transaction.csv").exists() else ROOT / "data/raw-sample"
OUT = ROOT / ("docs/data_profile.md" if RAW == REAL else "docs/data_profile_sample.md")
tx, ident = load_raw(RAW)
p = profile(tx, ident)
print("source:", RAW.relative_to(ROOT))

In [ ]:
pd.Series({k: p[k] for k in ["n_transactions", "n_fraud", "fraud_rate", "n_identity",
    "identity_coverage", "fraud_rate_with_identity", "fraud_rate_without_identity",
    "dt_min_days", "dt_max_days", "dt_span_days"]})

In [ ]:
pd.Series(p["missing"], name="missing_rate").sort_values(ascending=False).to_frame()

In [ ]:
display(pd.Series(p["top_device_info"], name="DeviceInfo").to_frame())
display(pd.Series(p["top_p_email"], name="P_emaildomain").to_frame())

In [ ]:
daily = tx.assign(day=(tx.TransactionDT // DAY).astype(int)).groupby("day").agg(n=("isFraud", "size"), fraud_rate=("isFraud", "mean"))
fig, ax = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
daily.n.plot(ax=ax[0], title="Transactions per day")
daily.fraud_rate.rolling(7).mean().plot(ax=ax[1], title="Fraud rate (7-day mean)")
plt.tight_layout()

In [ ]:
OUT.write_text(to_markdown(p, str(RAW.relative_to(ROOT))))
print("wrote", OUT.relative_to(ROOT))